<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks_evaluation_04_run_colab_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==============================================================================
# Phase 8: E2E Bass Transcription Pipeline Benchmark (Colab)
# [Cell 1] 환경 설정, GPU 할당 검증, 저장소 동기화 및 무결성 점검
# ==============================================================================
# 이 노트북은 Slakh2100 데이터셋(130 트랙)을 활용하여 음원 분리(Demucs) 및
# 미디 채보(CREPE) 파이프라인의 성능을 대규모로 평가합니다.
# ==============================================================================

import os
import sys
import shutil
import importlib.util
import subprocess
from google.colab import drive
import torch

# 1. 구글 드라이브 마운트
print("📂 구글 드라이브 마운트 중...")
drive.mount('/content/drive')

# 2. GPU 할당 검증 (필수)
if not torch.cuda.is_available():
    raise SystemError("❌ GPU가 할당되지 않았습니다. 상단 메뉴 [런타임] -> [런타임 유형 변경]에서 T4 GPU를 선택하십시오.")
print(f"✅ GPU 활성화됨: {torch.cuda.get_device_name(0)}")

# 3. 기존 폴더 초기화 및 깃허브 저장소 클론 (★ 본인 URL로 수정)
print("\n📦 저장소 클론 및 작업 공간 초기화 중...")
!rm -rf /content/Bass-separator
!git clone https://github.com/sjkim-audio/Bass-separator.git /content/Bass-separator

# 4. PYTHONPATH 영속성 확보 (os.environ 대신 sys.path 사용)
if "/content/Bass-separator" not in sys.path:
    sys.path.append("/content/Bass-separator")
%cd /content/Bass-separator

# 5. 시스템 의존성 점검 (FFmpeg 설치)
print("\n🔧 [시스템] 필수 도구 확인 중...")
if shutil.which("ffmpeg") is None:
    print("⚠️ FFmpeg가 없습니다. apt-get으로 설치합니다...")
    !apt-get update -qq
    !apt-get install -y ffmpeg -qq
else:
    print("✅ FFmpeg가 이미 설치되어 있습니다.")

# 6. 파이썬 요구 패키지 설치
print("\n🐍 [파이썬] 라이브러리 설치 중...")
!pip install -q -r requirements.txt
!pip install -q mir_eval museval pretty_midi

# 7. 설치 무결성 점검 (Health Check)
print("\n🏥 설치 무결성 점검 (Health Check)...")
critical_libs = ["demucs", "torchaudio", "librosa", "museval", "mir_eval"]
missing = [lib for lib in critical_libs if importlib.util.find_spec(lib) is None]

if not missing:
    print("✅ 필수 라이브러리가 모두 정상적으로 준비되었습니다!")
else:
    raise ImportError(f"❌ 다음 라이브러리가 누락되었습니다: {', '.join(missing)}")

📂 구글 드라이브 마운트 중...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ GPU 활성화됨: Tesla T4

📦 저장소 클론 및 작업 공간 초기화 중...
Cloning into '/content/Bass-separator'...
remote: Enumerating objects: 2185, done.
remote: Counting objects: 100% (321/321), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 2185 (delta 260), reused 196 (delta 186), pack-reused 1864 (from 1)
Receiving objects: 100% (2185/2185), 305.16 MiB | 26.81 MiB/s, done.
Resolving deltas: 100% (1357/1357), done.
/content/Bass-separator

🔧 [시스템] 필수 도구 확인 중...
✅ FFmpeg가 이미 설치되어 있습니다.

🐍 [파이썬] 라이브러리 설치 중...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 352.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

In [3]:
# ==============================================================================
# [Cell 2] 데이터셋 I/O 샌드박싱 및 압축 해제
# ==============================================================================
# 구글 드라이브의 네트워크 I/O 병목을 피하기 위해, 압축 파일을 코랩의
# 로컬 SSD 공간으로 복사하여 압축을 풉니다.
# ==============================================================================

import os

# 평가용 로컬 디렉토리 생성
os.makedirs("./slakh_processed/test", exist_ok=True)

# 드라이브의 zip 파일을 Colab 로컬 환경으로 조용히(-q) 압축 해제
# (★ 드라이브 내 slakh_test.zip의 실제 경로로 수정하십시오)
print("🗜️ 데이터셋 압축 해제 중 (수 분 소요될 수 있습니다)...")
!unzip -q /content/drive/MyDrive/Bass_separator/dataset/slakh_test.zip -d ./slakh_processed/test

# 데이터 무결성 검증 (정상 필터링 시 130 출력)
print("\n✅ 추출된 유효 트랙 수:")
!ls -1 ./slakh_processed/test | wc -l

🗜️ 데이터셋 압축 해제 중 (수 분 소요될 수 있습니다)...

✅ 추출된 유효 트랙 수:
130


In [ ]:
# ==============================================================================
# [Cell 3] 벤치마크 1: 채보 알고리즘 단독 평가 (Isolated Mode)
# ==============================================================================
# Demucs 분리 과정을 생략하고 정답 베이스 오디오(bass_gt.wav)를
# 직접 주입하여 피치 트래커와 양자화기의 순수 성능(Upper Bound)을 측정합니다.
# ==============================================================================

print("🚀 [1/2] 순수 미디 채보 성능 평가 시작 (Isolated Mode)...")

!PYTHONPATH=/content/Bass-separator python -m src.evaluation.run_batch_eval \
    --test_dir ./slakh_processed/test \
    --isolated True \
    --onset_tolerance 0.1 \
    --exp_id Colab_Phase8_Isolated

In [ ]:
# ==============================================================================
# [Cell 4] 벤치마크 2: 전체 파이프라인 성능 평가 (E2E Mode)
# ==============================================================================
# 믹스 오디오(mix.wav)를 주입하여 Demucs의 분리 성능(SDR, SIR, SAR)과
# 그로 인한 최종 채보 점수 하락폭(Gap)을 측정합니다.
# ==============================================================================

print("🚀 [2/2] E2E 전체 파이프라인 평가 시작 (Mix Mode)...")

!PYTHONPATH=/content/Bass-separator python -m src.evaluation.run_batch_eval \
    --test_dir ./slakh_processed/test \
    --isolated False \
    --onset_tolerance 0.1 \
    --exp_id Colab_Phase8_E2E

In [ ]:
# ==============================================================================
# [Cell 5] 평가 결과 영구 백업
# ==============================================================================
# Colab 세션이 종료되면 로컬 데이터가 소실되므로, 생성된 결과 JSON 리포트를
# 구글 드라이브의 안전 지대로 즉시 복사합니다.
# ==============================================================================
# [날짜 동적 생성] 현재 시스템 시간을 기반으로 yyyymmdd 형식의 문자열 생성
# ---------------------------------------------------------------------
current_date = datetime.now().strftime("%Y%m%d")
exp_id_with_date = f"Colab_Phase8_Isolated_{current_date}"

# (★ 백업할 드라이브 경로를 본인 환경에 맞게 수정하십시오)
destination_path = "/content/drive/MyDrive/Bass_separator/dataset/"

print(f"💾 결과 파일을 {destination_path} 경로로 백업합니다...")
!cp ./results/{exp_id_with_date}_batch_results.json {destination_path}

print("✅ 최종 평가 리포트 백업 완료.")